In [ ]:
!pip install kaggle

In [ ]:
import os
import getpass

# Pede para o usuário inserir as credenciais de forma segura
os.environ['KAGGLE_USERNAME'] = input()
os.environ['KAGGLE_KEY'] = getpass.getpass('API KEY: ')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import matplotlib.pyplot as plt
import numpy as np

# Definição dos caminhos para os diretórios de treino, validação e teste
train_dir = 'Vegetable Images/train'
validation_dir = 'Vegetable Images/validation'
test_dir = 'Vegetable Images/test'

# Parâmetros para as imagens e o treinamento
IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32

# Vamos verificar o número de classes (tipos de vegetais)
num_classes = len(os.listdir(train_dir))
print(f"Número de classes de vegetais encontrado: {num_classes}")

In [ ]:
# Criando instâncias de ImageDataGenerator
# Para o conjunto de treino, aplicamos data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normaliza os pixels para o intervalo [0, 1]
    rotation_range=40,          # Faixa de rotação aleatória em graus
    width_shift_range=0.2,      # Faixa de deslocamento horizontal
    height_shift_range=0.2,     # Faixa de deslocamento vertical
    shear_range=0.2,            # Intensidade do corte (shear)
    zoom_range=0.2,             # Faixa de zoom aleatório
    horizontal_flip=True,       # Inverte horizontalmente as imagens de forma aleatória
    fill_mode='nearest'         # Estratégia para preencher pixels criados após transformações
)

# Para os conjuntos de validação e teste, apenas normalizamos os dados. Não aplicamos augmentation.
test_datagen = ImageDataGenerator(rescale=1./255)

# Criando os geradores de dados que vão ler as imagens dos diretórios
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH), # Redimensiona todas as imagens para 150x150
    batch_size=BATCH_SIZE,
    class_mode='categorical' # Porque temos mais de 2 classes
)

validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# A saída nos mostrará quantas imagens e classes foram encontradas em cada conjunto.
# O resultado deve bater com as 15 classes que você encontrou.

In [ ]:
# Função para exibir imagens
def plot_images(images_arr):
    fig, axes = plt.subplots(1, 5, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# Pega um lote de imagens do gerador de treino
sample_training_images, _ = next(train_generator)

# Mostra as 5 primeiras imagens do lote
plot_images(sample_training_images[:5])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Inicializando o modelo sequencial
model = Sequential([
    # 1ª Camada de Convolução e Pooling
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(2, 2),

    # 2ª Camada de Convolução e Pooling
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # 3ª Camada de Convolução e Pooling
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    # Achatando os resultados para alimentar a parte densa da rede
    Flatten(),

    # Camada Densa com Dropout
    Dense(512, activation='relu'),
    Dropout(0.5), # "Desliga" 50% dos neurônios aleatoriamente para evitar overfitting

    # Camada de Saída
    Dense(num_classes, activation='softmax') # num_classes foi definido antes como 15
])

# Mostra um resumo da arquitetura do modelo
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Definindo o número de épocas
EPOCHS = 25

# Calculando os passos por época para treino e validação
steps_per_epoch = train_generator.n // train_generator.batch_size
validation_steps = validation_generator.n // validation_generator.batch_size

# Treinando o modelo
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_steps
)

In [ ]:
# Coletando os dados do histórico de treinamento
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

# Plotando a Acurácia de Treino e Validação
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Acurácia de Treino')
plt.plot(epochs_range, val_acc, label='Acurácia de Validação')
plt.legend(loc='lower right')
plt.title('Acurácia de Treino e Validação')

# Plotando a Perda de Treino e Validação
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Perda de Treino')
plt.plot(epochs_range, val_loss, label='Perda de Validação')
plt.legend(loc='upper right')
plt.title('Perda de Treino e Validação')
plt.show()

In [ ]:
# Primeiro, criamos um gerador para os dados de teste (apenas com normalização)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # É importante não embaralhar os dados de teste
)

# Avaliando o modelo no conjunto de teste
print("\n Avaliando o modelo no conjunto de teste...")
test_loss, test_acc = model.evaluate(test_generator)
print(f'\nAcurácia no conjunto de teste: {test_acc:.4f}')
print(f'Perda no conjunto de teste: {test_loss:.4f}')

In [ ]:
# Pegando um lote de imagens e rótulos do conjunto de teste
x_test, y_test = next(test_generator)

# Escolhendo uma imagem aleatória do lote para prever
image_to_predict = x_test[0:1] # Pegando a primeira imagem
true_label_index = np.argmax(y_test[0])

# Fazendo a previsão
prediction = model.predict(image_to_predict)

# Obtendo o índice da classe prevista (o que tiver a maior probabilidade)
predicted_class_index = np.argmax(prediction)

# Mapeando os índices para os nomes das classes (vegetais)
class_labels = list(train_generator.class_indices.keys())
predicted_class_name = class_labels[predicted_class_index]
true_class_name = class_labels[true_label_index]

# Mostrando a imagem e o resultado
plt.figure()
plt.imshow(image_to_predict[0])
plt.title(f"Verdadeiro: {true_class_name} | Previsto: {predicted_class_name}")
plt.axis('off')
plt.show()

# Opcional: ver a confiança da previsão
print(f"Confiança da previsão: {np.max(prediction)*100:.2f}%")

In [ ]:
# Para salvar
model.save('meu_classificador_vegetais.keras')

# Para carregar depois
# from tensorflow.keras.models import load_model
# modelo_carregado = load_model('meu_classificador_vegetais.h5')